# 🚢 AI Logistics & Finance Data Pipeline
### Version 2.0: Accountant Intelligence Edition
#### Batch Processing: OCR -> AI Structuring -> Finance Metrics -> Accountant CSV

This notebook has been updated to support the **Accountant Overview Dashboard**, extracting deep financial insights including Revenue, Expense Categorization, and Payment Status.

---

### 1. Environment Setup
Requirements:
- `pip install pandas pdfplumber pytesseract Pillow httpx python-dotenv`

In [ ]:
import os
import json
import uuid
import zipfile
import asyncio
import pandas as pd
import httpx
import pdfplumber
import pytesseract
from PIL import Image
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()

# CONFIGURATION
ZIP_PATH = "invoices.zip"  
EXTRACT_DIR = "temp_invoices/"
OPENROUTER_API_KEY = "YOUR_OPENROUTER_API_KEY"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR)

### 2. Advanced OCR & AI Engine

In [ ]:
def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    try:
        if ext == ".pdf":
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    content = page.extract_text()
                    if content: text += content + "\n"
        elif ext in [".jpg", ".jpeg", ".png"]:
            image = Image.open(file_path).convert("L")
            text = pytesseract.image_to_string(image)
    except Exception as e:
        print(f"OCR Error: {e}")
    return text.strip()

async def get_accountant_structure(raw_text):
    prompt = """
    Extract detailed Financial/Logistics data.
    Return ONLY valid JSON including:
    - product_name, vendor_name, quantity, unit_price, invoice_date, 
    - origin_country, destination_country, suggested_hsn_code,
    - expense_category (Options: 'Raw Materials', 'Logistics Fees', 'Office Rent', 'Utility'),
    - payment_status (Options: 'Paid', 'Pending')
    
    Format:
    {
      "product_name": "", "vendor_name": "", "quantity": 0, "unit_price": 0.0, 
      "invoice_date": "YYYY-MM-DD", "origin_country": "", "destination_country": "", 
      "hsn": "", "expense_category": "", "payment_status": ""
    }
    """
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": "openai/gpt-4o-mini",
        "messages": [{"role": "user", "content": f"{prompt}\n\nTEXT:\n{raw_text}"}]
    }
    async with httpx.AsyncClient() as client:
        resp = await client.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
        content = resp.json()["choices"][0]["message"]["content"]
        return json.loads(content.strip().replace("```json", "").replace("```", ""))

### 3. Financial Metrics Strategy

In [ ]:
def calculate_duty(value, hsn_prefix):
    rates = {"85": 0.18, "84": 0.15, "52": 0.12, "30": 0.05} # HSN logic
    rate = rates.get(hsn_prefix[:2], 0.10)
    duty = value * rate
    tax = (value + duty) * 0.18
    return duty, tax, rate

def assess_risk(origin, category):
    high_risk_origins = ["iran", "afghanistan", "north korea"]
    score = 15 if origin.lower() in high_risk_origins else 5
    if category == "Raw Materials": score += 10
    return score, ("High" if score > 20 else "Low")

### 4. Running the Financial Pipeline

In [ ]:
async def process_all():
    if os.path.exists(ZIP_PATH):
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf: zf.extractall(EXTRACT_DIR)
    
    dataset = []
    for filename in os.listdir(EXTRACT_DIR):
        path = os.path.join(EXTRACT_DIR, filename)
        print(f"-> Analyzing {filename}...")
        
        text = extract_text(path)
        if not text: continue
        
        fin_data = await get_accountant_structure(text)
        revenue = fin_data['quantity'] * fin_data['unit_price']
        duty, tax, d_rate = calculate_duty(revenue, fin_data['hsn'])
        risk_score, risk_lvl = assess_risk(fin_data['origin_country'], fin_data['expense_category'])
        
        dataset.append({
            "shipment_id": f"SHN-{uuid.uuid4().hex[:8].upper()}",
            "vendor": fin_data['vendor_name'],
            "product": fin_data['product_name'],
            "revenue": revenue,
            "duty_expense": duty,
            "tax_expense": tax,
            "total_landed_cost": revenue + duty + tax,
            "hsn_code": fin_data['hsn'],
            "category": fin_data['expense_category'],
            "status": fin_data['payment_status'],
            "invoice_date": fin_data['invoice_date'],
            "risk_score": risk_score,
            "risk_level": risk_lvl,
            "origin": fin_data['origin_country'],
            "destination": fin_data['destination_country']
        })
    
    return pd.DataFrame(dataset)

df = await process_all()
pd.set_option('display.max_columns', None)
df.head()

### 5. Final Export & Forecasting Analysis

In [ ]:
df.to_csv("logistics_accountant_dataset.csv", index=False)
print("Success: Created 'logistics_accountant_dataset.csv' ready for Dashboard population.")